In [3]:
import os
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import pickle
import json
import requests
from tqdm.notebook import tqdm
import ast
import gc
import re
import time
import os
from datetime import datetime, timedelta, timezone

Retrieve results for when CVE was modified in some way. The below code block dynamically determines what the start and end dates are by looking through previously saved folders, getting the most recently saved folder (folder names are just the end_date, so the state of the NVD at the end_date), and adding 120 days to it. If 120 days later is past the current date, then just use the current date.

In [4]:
SAVED_DIR = "saved"

def parse_folder_datetime(folder_name: str) -> datetime | None:
    try:
        iso_like = folder_name.replace("T", " ").replace("Z", "")
        date_part, time_part = iso_like.split(" ")

        h, m, s, ms = time_part.split("-")
        iso_string = f"{date_part}T{h}:{m}:{s}.{ms}Z"

        return datetime.strptime(
            iso_string, "%Y-%m-%dT%H:%M:%S.%fZ"
        ).replace(tzinfo=timezone.utc)

    except Exception:
        return None


def find_latest_saved_date() -> datetime:
    dates = []

    for name in os.listdir(SAVED_DIR):
        path = os.path.join(SAVED_DIR, name)

        if not os.path.isdir(path):
            continue

        # Only consider folders that look like dates
        if not name.startswith("2"):
            continue

        dt = parse_folder_datetime(name)
        if dt:
            dates.append(dt)

    return max(dates)


def to_iso_z(dt: datetime) -> str:
    return dt.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"

latest_date = find_latest_saved_date()
start_date_dt = latest_date

end_date_dt = start_date_dt + timedelta(days=120)

now_utc = datetime.now(timezone.utc)
if end_date_dt > now_utc:
    end_date_dt = now_utc

start_date = to_iso_z(start_date_dt)
end_date = to_iso_z(end_date_dt)

print("start_date =", start_date)
print("end_date =", end_date)


start_date = 2026-02-02T17:28:10.313Z
end_date = 2026-03-23T15:40:42.551Z


In [5]:
BASE_URL = "https://services.nvd.nist.gov/rest/json/cvehistory/2.0"
# these are the events to check for CWE modifications
event_types = [
    "CVE Modified",
    "CWE Remap",
    "Modified Analysis",
    "Reanalysis",
    "Initial Analysis",  # for new CVEs
]

results_per_page = 5000

# Store results per event type
all_changes = {event: [] for event in event_types}
for event in event_types:
    print(f"\nFetching events for: {event}")
    start_index = 0
    # maximum of 5000 results in a page, so must parse until reach all pages
    while True:
        params = {
            "eventName": event,
            "changeStartDate": start_date,
            "changeEndDate": end_date,
            "resultsPerPage": results_per_page,
            "startIndex": start_index
        }
        response = requests.get(BASE_URL, params=params)
        response.raise_for_status()
        data = response.json()
        cve_changes = data.get("cveChanges", [])
        total = data.get("totalResults", 0)
        all_changes[event].extend(cve_changes)
        print(
            f"  Retrieved {len(cve_changes)} events "
            f"(startIndex={start_index}, total={total})"
        )
        # Stop if we've retrieved all events
        start_index += results_per_page
        if start_index >= total:
            break
        time.sleep(6)  # necessary to not throttle api

print("\nDone fetching all event types.")



Fetching events for: CVE Modified
  Retrieved 4812 events (startIndex=0, total=4812)

Fetching events for: CWE Remap
  Retrieved 0 events (startIndex=0, total=0)

Fetching events for: Modified Analysis
  Retrieved 398 events (startIndex=0, total=398)

Fetching events for: Reanalysis
  Retrieved 224 events (startIndex=0, total=224)

Fetching events for: Initial Analysis
  Retrieved 5000 events (startIndex=0, total=6502)
  Retrieved 1502 events (startIndex=5000, total=6502)

Done fetching all event types.


In the cell above, we retrieve all changes. Below, we filter for when the CVE-CWE mappings change because that is what we include in the KG.

In [6]:
cwe_event_types = event_types[:-1]
cwe_changes = []
cpe_changes = []
count = 0
for event in cwe_event_types:
    for change in all_changes[event]:
        date = change['change']['created']
        all_c = change['change']['details']
        cve = change['change']['cveId']
        for c in all_c:
            # part where we only filter for CWE changes
            if c['type'] == 'CWE':
                cwe_changes.append((cve, c, date))
            # filter for CPE changes here
            if c['type'] == 'CPE Configuration':
                cpe_changes.append((cve, c, date))
    print(len(cwe_changes), len(cpe_changes))

1183 6
1183 6
1221 398
1224 589


In [7]:
changes_list = []
for change in cwe_changes:
    cveid = change[0]
    action = change[1]['action']
    date = change[2]
    if action == 'Added':
        cwe = change[1]['newValue']
    elif action == 'Removed':
        cwe = change[1]['oldValue']
    changes_list.append((cveid, action, cwe, date))
df_changes = pd.DataFrame(changes_list, columns=['CVE ID', 'Action', 'CWE', 'Date'])
safe_end_date = end_date.replace(":", "-").replace(".", "-")
# Create directory named after end_date
output_dir = os.path.join("saved", safe_end_date)
os.makedirs(output_dir, exist_ok=True)
csv_path = os.path.join(output_dir, "changes_cwe.csv")
df_changes.to_csv(csv_path, index=False)

Apply those changes to our current running list of CVEs. Save a new CSV file.

In [8]:
df_changes

,CVE ID,Action,CWE,Date
0,CVE-2025-63649,Added,CWE-125,2026-02-02T23:16:00.167
1,CVE-2025-71007,Added,CWE-20,2026-02-02T23:16:03.173
2,CVE-2025-71007,Removed,CWE-120,2026-02-02T23:16:03.173
3,CVE-2019-19006,Added,CWE-287,2026-02-03T15:16:08.983
4,CVE-2021-39935,Added,CWE-918,2026-02-03T15:16:11.330
...,...,...,...,...
1219,CVE-2025-38501,Removed,NVD-CWE-noinfo,2026-03-17T16:04:50.967
1220,CVE-2023-53392,Removed,CWE-668,2026-03-17T16:36:34.700
1221,CVE-2025-28164,Added,CWE-401,2026-03-04T19:42:07.987
1222,CVE-2026-23112,Added,CWE-787,2026-03-19T19:20:45.530


In [9]:
def parse_dt(dt_str):
    return datetime.fromisoformat(dt_str.replace("Z", "+00:00"))

In [10]:
df_changes = pd.read_csv(csv_path)
safe_start_date = start_date.replace(":", "-").replace(".", "-")

# get previous state of NVD using the start date
curr_cves = pd.read_csv(f'saved/{safe_start_date}/cve_metadata_{start_date}.csv')
changes_list = df_changes[['CVE ID', 'Action', 'CWE', 'Date']].values.tolist()

In [11]:
list_columns = ["matching_cwes", "matching_cpes", "keywords"]
# convert list representation of string into actual lists when reading from csv file
for col in list_columns:
    curr_cves[col] = curr_cves[col].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

To make the application of changes faster, we turn the list of current CVEs into dictionary. Then we iterate through all the changes to see which CVEs need to be modified, change those CVE entries, and then convert the dictionary of current CVEs back into a list, then into a Pandas dataframe, and then save a new CSV file. This should be considered the current CVE status when running updates again in the future.

In [12]:
curr_cves_list = curr_cves[curr_cves.columns.values.tolist()].values.tolist()

Sometimes, CWEs contain company information before the CWE. For example, Redhat Inc. CWE-### is the format of some CWE, so this helper function normalizes by just extracting the part of the CWE that contains CWE-####.

In [13]:
def normalize_cwe(cwe_str):
    if not cwe_str:
        return None
    match = re.search(r'CWE-\d+', cwe_str)
    return match.group(0) if match else None

In [14]:
curr_cves_dict = {}
for cve_entry in curr_cves_list:
    curr_cves_dict[cve_entry[0]] = cve_entry[1:]

for change in changes_list:
    cve = change[0]
    action = change[1]
    raw_cwe = change[2]
    date = change[3]
    cwe = normalize_cwe(raw_cwe)
    if cwe is None or cve not in curr_cves_dict:
        continue

    if action == "Added":
        curr_cves_dict[cve][2].append(cwe)

    elif action == "Removed":
        try:
            curr_cves_dict[cve][2].remove(cwe)
        except ValueError:
            continue
    curr_date = curr_cves_dict[cve][0]
    if parse_dt(date) > parse_dt(curr_date):
        curr_cves_dict[cve][0] = date


Next, we follow a similar process to update any CPEs

In [15]:
changes_list = []
for change in cpe_changes:
    cveid = change[0]
    try:
        action = change[1]['action']
    except KeyError:
        continue
    date = change[2]
    if action == 'Added':
        cwe = change[1]['newValue']
    elif action == 'Removed':
        cwe = change[1]['oldValue']
    changes_list.append((cveid, action, cwe, date))
df_changes = pd.DataFrame(changes_list, columns=['CVE ID', 'Action', 'CWE', 'Date'])
safe_end_date = end_date.replace(":", "-").replace(".", "-")
# Create directory named after end_date
output_dir = os.path.join("saved", safe_end_date)
os.makedirs(output_dir, exist_ok=True)
csv_path = os.path.join(output_dir, "changes_cpe.csv")
print(csv_path)
df_changes.to_csv(csv_path, index=False)

saved/2026-03-23T15-40-42-551Z/changes_cpe.csv


In [16]:
CPE_PATTERN = re.compile(r"cpe:2\.3:[aho]:[^\s*]+")

def extract_cpes_from_change(text):
    if not text:
        return []
    return list(set(CPE_PATTERN.findall(text)))

def normalize_cpe(cpe):
    return cpe.strip().lower()

In [17]:
for change in cpe_changes:
    cve = change[0]
    meta = change[1]
    date = change[2]

    if cve not in curr_cves_dict:
        continue

    action = meta.get("action")
    cpe_list = curr_cves_dict[cve][4]

    if action == "Added":
        cpes = extract_cpes_from_change(meta.get("newValue"))

        for raw_cpe in cpes:
            cpe = normalize_cpe(raw_cpe)
            if cpe not in cpe_list:
                cpe_list.append(cpe)

    elif action == "Removed":
        cpes = extract_cpes_from_change(meta.get("oldValue"))

        for raw_cpe in cpes:
            cpe = normalize_cpe(raw_cpe)
            try:
                cpe_list.remove(cpe)
            except ValueError:
                continue

    # Update last modified date
    curr_date = curr_cves_dict[cve][0]
    if parse_dt(date) > parse_dt(curr_date):
        curr_cves_dict[cve][0] = date

Above, we've added all the changes to CWE and CPE. But we also need to add new CVEs using the Initial Analysis event results from the API.

In [18]:
init_analysis = all_changes['Initial Analysis']
additions = []
for new in init_analysis:
    cveid = new['change']['cveId']
    details = new['change']['details']
    date = new['change']['created']
    cwes = []
    cpes = []
    for action in details:
        if action['type'] == 'CWE':
            cwes.append(action['newValue'])
        elif action['type'] == 'CPE Configuration':
            try:
                cpes.extend(extract_cpes_from_change(action['newValue']))
            except KeyError:
                continue
    additions.append((cveid, date, '', cwes, [], cpes))

In [19]:
new_additions = []
for addition in additions:
    if addition[0] not in curr_cves_dict:
        new_additions.append(addition)

In [20]:
curr_cves_list = []
for key in curr_cves_dict.keys():
    date, description, cwe, keywords, cpe = curr_cves_dict[key]
    curr_cves_list.append((key, date, description, cwe, keywords, cpe))
curr_cves_list.extend(new_additions)
columns = [
    "cve_id",
    "last_modified",
    "description",
    "matching_cwes",
    "keywords",
    "matching_cpes"
]
curr_cves = pd.DataFrame(curr_cves_list, columns=columns)
path_to_save = f"{output_dir}/cve_metadata_{end_date}.csv"
curr_cves.to_csv(path_to_save, index=False)